## Sample the data for inspection

In [1]:
import pandas as pd

# Load data from transformed folder (using latest version t03)
train = pd.read_parquet("../../data/transformed/train_t03.parquet")
test = pd.read_parquet("../../data/transformed/test_t03.parquet")
val = pd.read_parquet("../../data/transformed/val_t03.parquet")

# Create sample data (10% of each split)
train_sample = train.sample(frac=0.1, random_state=42)
test_sample = test.sample(frac=0.1, random_state=42)
val_sample = val.sample(frac=0.1, random_state=42)

print("=== TRAIN SAMPLE SHAPE ===")
print(f"Original: {train.shape}, Sample: {train_sample.shape}")

print("\n=== TEST SAMPLE SHAPE ===")
print(f"Original: {test.shape}, Sample: {test_sample.shape}")

print("\n=== VAL SAMPLE SHAPE ===")
print(f"Original: {val.shape}, Sample: {val_sample.shape}")

print("\n=== FIRST 5 ROWS (TRAIN SAMPLE) ===")
print(train_sample.head())

print("\n=== COLUMN TYPES ===")
print(train_sample.dtypes)

print("\n=== TARGET DISTRIBUTION ===")
target_candidates = ["Results", "Result", "violations_recorded", "Violation", "inspection_result"]
target_col = next((col for col in target_candidates if col in train_sample.columns), None)
if target_col is not None:
    print(f"Using target column: {target_col}")
    print(train_sample[target_col].value_counts())
else:
    print(f"No target column found. Available columns: {list(train_sample.columns)}")

print("\n=== MISSING VALUES ===")
print(train_sample.isnull().sum()[train_sample.isnull().sum() > 0])

=== TRAIN SAMPLE SHAPE ===
Original: (109693, 17), Sample: (10969, 17)

=== TEST SAMPLE SHAPE ===
Original: (34294, 17), Sample: (3429, 17)

=== VAL SAMPLE SHAPE ===
Original: (27483, 17), Sample: (2748, 17)

=== FIRST 5 ROWS (TRAIN SAMPLE) ===
       Risk  Results  has_prior_inspection  violation_count  inspection_year  \
19729     3        0                  True                1             2015   
17969     3        0                  True                3             2015   
42072     3        0                  True                3             2012   
34509     2        0                  True                0             2013   
77267     1        1                  True                3             2011   

       inspection_month  inspection_dayofweek  inspection_quarter  \
19729                10                     3                   4   
17969                 4                     4                   2   
42072                 5                     0                   2  

## Imports

In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, roc_auc_score, confusion_matrix,
    classification_report
)

## Data Loading

In [3]:
TRAIN_PATH = "../../data/transformed/train_t03.parquet"
VAL_PATH   = "../../data/transformed/val_t03.parquet"
TEST_PATH  = "../../data/transformed/test_t03.parquet"

TARGET   = "Results"

train = pd.read_parquet(TRAIN_PATH)
val   = pd.read_parquet(VAL_PATH)
test  = pd.read_parquet(TEST_PATH)

FEATURES = [col for col in train.columns if col != TARGET]

X_train, y_train = train[FEATURES], train[TARGET]
X_val,   y_val   = val[FEATURES],   val[TARGET]
X_test,  y_test  = test[FEATURES],  test[TARGET]

print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")
print(f"\nTarget distribution (train):\n{y_train.value_counts()}")

Train: (109693, 16) | Val: (27483, 16) | Test: (34294, 16)

Target distribution (train):
Results
0    77306
1    32387
Name: count, dtype: int64


## Hyperparamters Definition

In [4]:
# These are the parameters that will be logged to MLflow later
# Tuned for a 70/30 class imbalance (Pass/Fail)
params = {
    "n_estimators":      200,
    "max_depth":         15,
    "min_samples_split": 5,
    "min_samples_leaf":  2,
    "max_features":      "sqrt",
    "class_weight":      "balanced",
    "random_state":      42,
    "n_jobs":            -1,
}

## Training

In [ ]:
model = RandomForestClassifier(**params)
model.fit(X_train, y_train)

print("Model trained successfully.")
print(f"Number of trees: {model.n_estimators}")

## Validation

In [ ]:
y_val_pred = model.predict(X_val)
y_val_prob = model.predict_proba(X_val)[:, 1]

val_cm = confusion_matrix(y_val, y_val_pred)
tn, fp, fn, tp = val_cm.ravel()

val_metrics = {
    # Standard metrics
    "val_accuracy":  accuracy_score(y_val, y_val_pred),
    "val_f1":        f1_score(y_val, y_val_pred),
    "val_roc_auc":   roc_auc_score(y_val, y_val_prob),
    "val_precision": precision_score(y_val, y_val_pred),
    "val_recall":    recall_score(y_val, y_val_pred),
    # Business metrics
    # Restaurants wrongly predicted to pass but actually fail = public health risk
    "val_false_negative_rate":        fn / (fn + tp),
    # Of all failing restaurants, how many did we correctly flag?
    "val_failing_catch_rate":         tp / (tp + fn),
}

print("=== VALIDATION METRICS ===")
for k, v in val_metrics.items():
    print(f"  {k:<35} {v:.4f}")

print(f"\nConfusion Matrix (Val):\n{val_cm}")
print(f"\nClassification Report (Val):\n")
print(classification_report(y_val, y_val_pred, target_names=["Pass (0)", "Fail (1)"]))

## Testing

In [ ]:
y_test_pred = model.predict(X_test)
y_test_prob = model.predict_proba(X_test)[:, 1]

test_cm = confusion_matrix(y_test, y_test_pred)
tn, fp, fn, tp = test_cm.ravel()

test_metrics = {
    # Standard metrics
    "test_accuracy":  accuracy_score(y_test, y_test_pred),
    "test_f1":        f1_score(y_test, y_test_pred),
    "test_roc_auc":   roc_auc_score(y_test, y_test_prob),
    "test_precision": precision_score(y_test, y_test_pred),
    "test_recall":    recall_score(y_test, y_test_pred),
    # Business metrics
    "test_false_negative_rate":  fn / (fn + tp),
    "test_failing_catch_rate":   tp / (tp + fn),
}

print("=== TEST METRICS ===")
for k, v in test_metrics.items():
    print(f"  {k:<35} {v:.4f}")

print(f"\nConfusion Matrix (Test):\n{test_cm}")
print(f"\nClassification Report (Test):\n")
print(classification_report(y_test, y_test_pred, target_names=["Pass (0)", "Fail (1)"]))

## Feature Importance

In [ ]:
importance_df = pd.DataFrame({
    "feature":    FEATURES,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False).reset_index(drop=True)

print("=== FEATURE IMPORTANCES ===")
print(importance_df.to_string(index=False))

## Exporting To MLflow

In [ ]:
# This cell does NOT log to MLflow directly.
# The central experiment_tracking.py file will import and log these.
# Just verify everything looks correct before handing off.

rf_output = {
    "model":        model,
    "params":       params,
    "val_metrics":  val_metrics,
    "test_metrics": test_metrics,
    "features":     FEATURES,
    "importance":   importance_df,
}

print("Random Forest output ready for experiment_tracking.py")
print(f"Val F1:  {val_metrics['val_f1']:.4f}")
print(f"Test F1: {test_metrics['test_f1']:.4f}")